In [1]:
import sys
import os

# Adjust the path to where the src folder is located
sys.path.append(os.path.abspath(os.path.join('..', 'src')))

In [2]:
import torch.nn as nn
import torch.utils.data.dataloader
from torch.nn.functional import scaled_dot_product_attention
import numpy as np
from language_models.utils import repackage_hidden, get_batch, batchify, save_checkpoint, move_to_device, save_val_loss_data
from language_models.dictionary_corpus import Corpus
from tqdm import tqdm
import math
import torch.nn.functional as F
import torch.optim as optim

# Model

In [3]:
def scaled_dot_product_attention_1(query, key, value, attn_mask=None, dropout_p=0.0,
    is_causal=False, scale=None, enable_gqa=False) -> torch.Tensor:

    L, S = query.size(-2), key.size(-2)

    scale_factor = 1 / math.sqrt(query.size(-1)) if scale is None else scale
    attn_bias = torch.zeros(L, S, dtype=query.dtype, device=query.device)
    if is_causal:
        assert attn_mask is None
        temp_mask = torch.ones(L, S, dtype=torch.bool).tril(diagonal=0)
        attn_bias.masked_fill_(temp_mask.logical_not(), float("-inf"))
        attn_bias.to(query.dtype)
        print(attn_bias)

    if attn_mask is not None:
        if attn_mask.dtype == torch.bool:
            attn_bias.masked_fill_(attn_mask.logical_not(), float("-inf"))
        else:
            attn_bias = attn_mask + attn_bias

    if enable_gqa:
        key = key.repeat_interleave(query.size(-3)//key.size(-3), -3)
        value = value.repeat_interleave(query.size(-3)//value.size(-3), -3)

    attn_weight = query @ key.transpose(-2, -1) * scale_factor
    attn_weight += attn_bias
    attn_weight = torch.softmax(attn_weight, dim=-1)
    attn_weight = torch.dropout(attn_weight, dropout_p, train=True)
    return attn_weight @ value

In [4]:
class CBR_RNN(nn.Module): 
# goal here is to reuse CBR_RNN but with scaled dot product attention for more efficient computations. 
# Also I got rid of options such as loading pretrained embeddings, and ablating attention to simplify the code.
# In the future if those options are needed, they can still be copy pasted from William's code as the structure hasn't changed
    def __init__(self, ntoken, ninp, nhid, nheads, dropout=0.5, device=None):
        super().__init__()
        #same layers as Timkey
        self.device = device
        self.nheads=nheads
        self.tanh = nn.Tanh()
        self.drop = nn.Dropout(dropout)
        self.score_attn = nn.Softmax(dim=-1)
        self.encoder = nn.Embedding(ntoken, ninp)
        self.q = nn.Linear(ninp+nhid,nhid)
        self.intermediate_h = nn.Linear(nhid*4,nhid*4)
        self.decoder = nn.Linear(nhid, ntoken)
        self.q_norm = torch.nn.LayerNorm(nhid)
        self.int_norm = torch.nn.LayerNorm(nhid * 4)
        self.f_norm = torch.nn.LayerNorm(nhid * 3)  
        self.nhid = nhid
        self.final_h = nn.Linear(nhid*4,nhid*3)
        self.multihead_attn = nn.MultiheadAttention(embed_dim=nhid, num_heads=nheads, batch_first=True)
        
        
        self.init_weights()
        
    def init_weights(self):
        """Initialize model weights for better training dynamics"""
        # General initialization
        for name, param in self.named_parameters():
            if 'weight' in name:
                if 'norm' in name:
                    nn.init.ones_(param)
                elif 'encoder' in name:
                    nn.init.normal_(param, mean=0, std=0.01)
                elif 'decoder' in name:
                    nn.init.normal_(param, mean=0, std=0.01)
                else:
                    # Standard He initialization for processing layers
                    nn.init.kaiming_normal_(param, mode='fan_in', nonlinearity='tanh')
            elif 'bias' in name:
                nn.init.zeros_(param)

    def init_cache(self, observation, nheads):#plus tard : rajouter un paramètre nheads 
        """Initialize hidden state and attention caches with better initialization strategy"""
        if len(observation.size()) > 1:
            bsz = observation.size(dim=-1)
        else:
            bsz = 1
            
        hidden = torch.zeros(1, bsz, self.nhid).to(self.device)*0.01
        if nheads ==1 :
            key_cache = torch.zeros(bsz, 1, 1, self.nhid).to(self.device) * 0.01
            value_cache = torch.zeros(bsz, 1, 1, self.nhid).to(self.device) * 0.01
        else:
            key_cache = torch.zeros(bsz, 1, self.nhid).to(self.device) * 0.01
            value_cache = torch.zeros(bsz, 1, self.nhid).to(self.device) * 0.01
        return hidden, key_cache, value_cache
        
    
    def forward(self, observation, initial_cache, nheads):
        # Get dimensions
        seq_len = observation.size(0) #if len(observation.size()) > 1 else 1
        # Unpack initial cache
        hidden, key_cache, value_cache = initial_cache

        # 1. Encode observations
        emb = self.drop(self.encoder(observation))
        # Process sequence : is there another more efficient way to compute causal attention than looping ?
        for i in range(seq_len): #need to keep sequential processing as the core structure is recurrent (each new word needs the hidden state obtained after prediction of the last word)
            # 2. Concatenate with previous hidden state
            combined = torch.cat((emb[i], hidden[-1]), -1)
            query = self.drop(self.tanh(self.q_norm(self.q(combined))))      
            query = query.unsqueeze(1)
            if nheads==1: 
                query = query.unsqueeze(1)
                attn_output = scaled_dot_product_attention(
                    query, key_cache, value_cache,#batch dimension needs to be the first one, hence the transpose (and the unsuqeeze on the query), second dim is seq len
                    is_causal=False
                )
                #squeeze head and seqlen dimensions (both are equal to 1 in this case anyways)
                attn = attn_output.squeeze(1).squeeze(1)
                query = query.squeeze(1).squeeze(1)
            else : 
                attn_output,_ = self.multihead_attn(query, key_cache, value_cache, is_causal = False)
                attn = attn_output.squeeze(1)
                query = query.squeeze(1)
                
                
            intermediate_input = torch.cat((emb[i], query, attn, hidden[-1]), -1)
            intermediate = self.drop(self.tanh(self.int_norm(self.intermediate_h(intermediate_input))))
            final_output = self.drop(self.tanh(self.f_norm(self.final_h(intermediate))))
            key_cache_i, value_cache_i, hidden_i = final_output.split(self.nhid, dim=-1)
            
            hidden_i = hidden_i.unsqueeze(0)
            hidden = torch.cat((hidden, hidden_i), dim=0)#hidden is [source seq len, batch size, hidden dim] -> easier to call hidden[i] when computing query

            if nheads==1:
                key_cache_i = key_cache_i.unsqueeze(1).unsqueeze(1)
                value_cache_i = value_cache_i.unsqueeze(1).unsqueeze(1)
                key_cache = torch.cat((key_cache, key_cache_i), dim=2)#concatenating along source sequence length dimension
                value_cache = torch.cat((value_cache, value_cache_i), dim=2)#key and value cache are [batch_size, nheads, source seq len, hidden_dim]
            
            else : 
                key_cache_i = key_cache_i.unsqueeze(1)
                value_cache_i = value_cache_i.unsqueeze(1)

                key_cache = torch.cat((key_cache, key_cache_i), dim=1)#concatenating along source sequence length dimension
                value_cache = torch.cat((value_cache, value_cache_i), dim=1)#key and value cache are [batch_size, source seq len, hidden_dim]
            
        decoded = self.decoder(hidden[1:])
        
        return decoded, hidden

In [21]:
0.001 == 1e-3

True

# Multihead

In [5]:
class Multihead_CBR_RNN(nn.Module): 
# goal here is to reuse CBR_RNN but with scaled dot product attention for more efficient computations. 
# Also I got rid of options such as loading pretrained embeddings, and ablating attention to simplify the code.
# In the future if those options are needed, they can still be copy pasted from William's code as the structure hasn't changed
    def __init__(self, ntoken, ninp, nhid, nheads, dropout=0.5, device=None):
        super().__init__()
        #same layers as Timkey
        self.device = device
        self.tanh = nn.Tanh()
        self.drop = nn.Dropout(dropout)
        self.score_attn = nn.Softmax(dim=-1)
        self.encoder = nn.Embedding(ntoken, ninp)
        self.q = nn.Linear(ninp+nhid,nhid)
        self.intermediate_h = nn.Linear(nhid*4,nhid*4)
        self.decoder = nn.Linear(nhid, ntoken+1)
        self.q_norm = torch.nn.LayerNorm(nhid)
        self.int_norm = torch.nn.LayerNorm(nhid * 4)
        self.f_norm = torch.nn.LayerNorm(nhid * 3)  
        self.nhid = nhid
        self.attn_div_factor = np.sqrt(nhid)
        self.final_h = nn.Linear(nhid*4,nhid*3)
        self.multihead_attn = nn.MultiheadAttention(embed_dim=nhid, num_heads=nheads, batch_first=True)
        self.nheads = nheads
        print(ninp)
        
    #same weight initialization as Timkey
    def init_weights(self, freeze_embedding, aux_objective):
        """ Initialize encoder and decoder weights """
        initrange = 0.1
        if not freeze_embedding:
            self.encoder.weight.data.uniform_(-initrange, initrange)
        self.decoder.bias.data.fill_(0)
        self.decoder.weight.data.uniform_(-initrange, initrange)
        if(aux_objective):
            self.aux_decoder.bias.data.fill_(0)
            self.aux_decoder.weight.data.uniform_(-initrange, initrange)

    
    def init_hidden(self, bsz):
        """ Initialize a fresh hidden state """
        weight = next(self.parameters()).data
    
        return torch.tensor(weight.new(bsz, self.nhid).zero_())
    
    def init_cache(self, observation):
        if len(observation.size())>1:
            bsz = observation.size(dim=-1)
        else:
            bsz = 1
        seq_len = observation.size(dim=0)

        return torch.zeros(1, bsz, self.nhid).to(self.device), torch.zeros(1, bsz, self.nhid).to(self.device), torch.zeros(1, bsz, self.nhid).to(self.device)


    def forward(self, observation, initial_cache, nheads):
        print(observation.shape)
        # Get dimensions
        seq_len = observation.size(0) #if len(observation.size()) > 1 else 1
        # Unpack initial cache
        hidden, key_cache, value_cache = initial_cache
        seq = hidden.size(0)
        batch = hidden.size(1)
        nhid = hidden.size(2)
        # 1. Encode observations
        emb = self.drop(self.encoder(observation))
        # Process sequence : is there another more efficient way to compute causal attention than looping ?
        for i in range(seq_len): #need to keep sequential processing as the core structure is recurrent (each new word needs the hidden state obtained after prediction of the last word)
            # 2. Concatenate with previous hidden state
            query = self.drop(self.tanh(self.q_norm(self.q(torch.cat((emb[i],hidden[i]), -1))))) #b * d
            query = query.unsqueeze(1) 
            #here multihead attention demands that the query be split wrt nb of heads. each split is processed in parallel, and the results are concatenated at the end.
            attn_output = self.multihead_attn(
                query, key_cache.transpose(0, 1), value_cache.transpose(0, 1)
                
            )
            #outputs attn output and attn output weights
            attn = attn_output[0]#.squeeze(1)
            intermediate = self.drop(self.tanh(self.int_norm(self.intermediate_h(torch.cat((emb[i].unsqueeze(0),query.transpose(0, 1),attn.transpose(0, 1),hidden[i].unsqueeze(0)),-1)))))
            intermediate_2 = self.drop(self.tanh(self.f_norm(self.final_h(intermediate))))
            key_cache_i, value_cache_i, hidden_i = intermediate_2.split(self.nhid, dim=-1)
            hidden = torch.cat((hidden, hidden_i), dim=0)
            key_cache = torch.cat((key_cache, key_cache_i), dim=0)
            value_cache = torch.cat((value_cache, value_cache_i), dim=0)
        decoded = self.decoder(hidden[1:])
        
        return decoded, hidden

# Training function

In [18]:
def train(model, criterion, train_data, batch_size, ntokens, nheads):
    # Turn on training mode which enables dropout.
    model.train()
    total_loss = 0
    #NEW : move hidden to devide
    #hidden = model.init_hidden(batch_size)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    first_batch, _ = get_batch(train_data, 0, 35)
    cache = model.init_cache(first_batch, nheads)
    for batch, i in enumerate(tqdm(range(0, train_data.size(0) - 1, 35), desc="Training")):
        data, targets = get_batch(train_data, i, 35)
        #NEW : move data and target to device
        
        # truncated BPP
        #hidden = repackage_hidden(hidden)
        model.zero_grad()
        output, hidden = model(data, cache, nheads)
        print('output.shape', output.shape)
        output_flat = output.reshape(-1, output.size(-1))
        print(output_flat.shape)
        # Similarly, reshape targets to [seq_len*batch_size]
        print('targets.shape', targets.shape)
        targets_flat = targets.reshape(-1)
        print(targets_flat.shape)
        #loss = criterion(output.view(-1, ntokens), targets)
        loss=criterion(output_flat, targets_flat)
        loss.backward()
        

        # `clip_grad_norm` helps prevent the exploding gradient problem in RNNs / LSTMs.
        torch.nn.utils.clip_grad_norm_(model.parameters(), 0.25)
        optimizer.step()
        print('ok')
        total_loss += loss.item()

# Data

In [6]:
corpus = Corpus('/scratch2/mrenaudin/colorlessgreenRNNs/english_data')


In [7]:
ntokens = len(corpus.dictionary)


In [8]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'


In [9]:
train_data = batchify(corpus.train, 128, device)#batch size of 128
val_data = batchify(corpus.valid, 128, device)
test_data = batchify(corpus.test, 128, device)

In [10]:
criterion = nn.CrossEntropyLoss()


In [11]:
model = CBR_RNN(ntokens, 12, 12,1) #ntokens, embedding size, nb of hidden units per layer

In [12]:
batch_size = 128

In [19]:
train(model, criterion, train_data, batch_size, ntokens,1)
#in CBR_RNN : hidden, key_cache, and value_chache are [seq_len, batch_size, nb of hidden units per layer]

Training:   0%|          | 0/18540 [00:00<?, ?it/s]

output.shape torch.Size([35, 128, 50001])
torch.Size([4480, 50001])
targets.shape torch.Size([4480])
torch.Size([4480])


Training:   0%|          | 1/18540 [00:00<3:06:15,  1.66it/s]

ok
output.shape torch.Size([35, 128, 50001])
torch.Size([4480, 50001])
targets.shape torch.Size([4480])
torch.Size([4480])


Training:   0%|          | 2/18540 [00:01<3:32:19,  1.46it/s]

ok


KeyboardInterrupt: 

In [56]:
L = 1
S = 12
temp_mask = torch.ones(L, S, dtype=torch.bool).tril(diagonal=0)
print('temp_mask', temp_mask)

temp_mask tensor([[ True, False, False, False, False, False, False, False, False, False,
         False, False]])


In [58]:
attn_bias = torch.zeros(L,S)
attn_bias

tensor([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.]])

In [59]:
attn_bias.masked_fill_(temp_mask.logical_not(), float("-inf"))


tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf, -inf]])